# 12a WILDTRACK MVDeTr Training

This notebook follows the actual MVDeTr repository layout and training entry point.

- Training entry point: `main.py`
- Dataset contract: `~/Data/Wildtrack`
- Expected WILDTRACK content: `Image_subsets/`, `annotations_positions/`, `calibrations/`
- CUDA op build location: `multiview_detector/models/ops`
- Fallback path: if the CUDA extension fails to build, the notebook patches MVDeTr to use the repo's slower PyTorch deformable-attention implementation instead of crashing

In [ ]:
import json
import platform
import sys
import traceback
from pathlib import Path

STATE_PATH = Path("/kaggle/working/12a_wildtrack_mvdetr_state.json")
state = {
    "repo_url": "https://github.com/hou-yz/MVDeTr.git",
    "repo_dir": "/kaggle/working/MVDeTr",
    "artifact_root": "/kaggle/working/outputs/12a_wildtrack_mvdetr",
    "wildtrack_candidates": [
        "/kaggle/input/large-scale-multicamera-detection-dataset",
        "/kaggle/input/datasets/aryashah2k/large-scale-multicamera-detection-dataset",
        "/kaggle/input/wildtrack",
        "/kaggle/input/datasets/aryashah2k/wildtrack",
    ],
    "weights_candidates": [
        "/kaggle/input/mtmc-weights",
        "/kaggle/input/datasets/mrkdagods/mtmc-weights",
    ],
}

try:
    import torch

    state["python"] = sys.version
    state["platform"] = platform.platform()
    state["torch_version"] = torch.__version__
    state["torch_cuda_version"] = torch.version.cuda
    state["cuda_available"] = bool(torch.cuda.is_available())
    state["gpu_name"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
except Exception:
    state["python"] = sys.version
    state["platform"] = platform.platform()
    state["torch_probe_error"] = traceback.format_exc()

STATE_PATH.parent.mkdir(parents=True, exist_ok=True)
STATE_PATH.write_text(json.dumps(state, indent=2), encoding="utf-8")
print(json.dumps(state, indent=2))

In [ ]:
import json
import re
import subprocess
import sys
import traceback
from pathlib import Path

STATE_PATH = Path("/kaggle/working/12a_wildtrack_mvdetr_state.json")
state = json.loads(STATE_PATH.read_text(encoding="utf-8"))
repo_dir = Path(state["repo_dir"])


def run(cmd, cwd=None):
    print("+", " ".join(map(str, cmd)))
    return subprocess.run(cmd, cwd=cwd, check=True)


def patch_file(path: Path, replacements):
    text = path.read_text(encoding="utf-8")
    original = text
    applied = []
    for pattern, repl, label in replacements:
        text, count = re.subn(pattern, repl, text, flags=re.MULTILINE | re.DOTALL)
        applied.append({"label": label, "count": count})
    if text != original:
        path.write_text(text, encoding="utf-8")
    return applied


summary = {}
try:
    if repo_dir.exists() and (repo_dir / ".git").exists():
        run(["git", "-C", str(repo_dir), "fetch", "--depth", "1", "origin"])
        run(["git", "-C", str(repo_dir), "checkout", "--force", "FETCH_HEAD"])
    elif not repo_dir.exists():
        run(["git", "clone", "--depth", "1", state["repo_url"], str(repo_dir)])
    else:
        print(f"Reusing existing directory: {repo_dir}")

    run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools", "wheel"])
    run([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "kornia",
        "opencv-python-headless",
        "scipy",
        "matplotlib",
        "pillow",
        "tqdm",
    ])

    patches = {}
    patches["ms_deform_attn_func.py"] = patch_file(
        repo_dir / "multiview_detector" / "models" / "ops" / "functions" / "ms_deform_attn_func.py",
        [
            (
                r"^import MultiScaleDeformableAttention as MSDA$",
                "try:\n    import MultiScaleDeformableAttention as MSDA\nexcept ImportError:\n    MSDA = None",
                "optional_cuda_import",
            ),
        ],
    )
    patches["ms_deform_attn.py"] = patch_file(
        repo_dir / "multiview_detector" / "models" / "ops" / "modules" / "ms_deform_attn.py",
        [
            (
                r"^from \.\.functions import MSDeformAttnFunction$",
                "from ..functions import MSDeformAttnFunction\nfrom ..functions.ms_deform_attn_func import ms_deform_attn_core_pytorch",
                "import_pytorch_fallback",
            ),
            (
                r"^        output = MSDeformAttnFunction\.apply\(value, input_spatial_shapes, input_level_start_index,\n                                            sampling_locations, attention_weights, self\.im2col_step\)$",
                "        try:\n"
                "            output = MSDeformAttnFunction.apply(\n"
                "                value, input_spatial_shapes, input_level_start_index,\n"
                "                sampling_locations, attention_weights, self.im2col_step)\n"
                "        except Exception as exc:\n"
                "            warnings.warn(f'Falling back to slower PyTorch ms_deform_attn implementation: {exc}')\n"
                "            output = ms_deform_attn_core_pytorch(\n"
                "                value, input_spatial_shapes, sampling_locations, attention_weights)",
                "runtime_fallback",
            ),
        ],
    )
    patches["Wildtrack.py"] = patch_file(
        repo_dir / "multiview_detector" / "datasets" / "Wildtrack.py",
        [
            (r"dtype=np\.float([^0-9a-zA-Z_])", r"dtype=float\1", "numpy_float_alias"),
        ],
    )
    patches["frameDataset.py"] = patch_file(
        repo_dir / "multiview_detector" / "datasets" / "frameDataset.py",
        [
            (r"dtype=np\.bool\)", "dtype=np.bool_)", "numpy_bool_alias"),
            (r"(?<!geometry\.transform\.)kornia\.warp_perspective", "kornia.geometry.transform.warp_perspective", "kornia_warp_perspective_namespace"),
            (r"(?<!geometry\.transform\.)kornia\.get_rotation_matrix2d", "kornia.geometry.transform.get_rotation_matrix2d", "kornia_get_rotation_matrix2d_namespace"),
        ],
    )

    readme_path = repo_dir / "README.md"
    readme = readme_path.read_text(encoding="utf-8", errors="ignore")
    summary = {
        "repo_ready": True,
        "repo_root_entries": sorted(path.name for path in repo_dir.iterdir()),
        "has_requirements_txt": (repo_dir / "requirements.txt").exists(),
        "ops_build_script": str(repo_dir / "multiview_detector" / "models" / "ops" / "make.sh"),
        "readme_mentions": {
            "train_command": "python main.py -d wildtrack" in readme,
            "data_home": "~/Data/" in readme,
            "mask_sh_mentioned": "mask.sh" in readme,
        },
        "patches": patches,
    }
except Exception:
    summary = {
        "repo_ready": False,
        "repo_setup_error": traceback.format_exc(),
    }

state.update(summary)
STATE_PATH.write_text(json.dumps(state, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))

In [ ]:
import json
import os
import re
import subprocess
import sys
import traceback
from pathlib import Path

STATE_PATH = Path("/kaggle/working/12a_wildtrack_mvdetr_state.json")
state = json.loads(STATE_PATH.read_text(encoding="utf-8"))
repo_dir = Path(state["repo_dir"])
ops_dir = repo_dir / "multiview_detector" / "models" / "ops"


def probe_model_import():
    env = os.environ.copy()
    env["PYTHONPATH"] = f"{repo_dir}:{env.get('PYTHONPATH', '')}" if env.get("PYTHONPATH") else str(repo_dir)
    return subprocess.run(
        [
            sys.executable,
            "-c",
            "import sys; sys.path.insert(0, r'%s'); from multiview_detector.models.mvdetr import MVDeTr; print('mvdetr import ok')"
            % repo_dir,
        ],
        cwd=repo_dir,
        env=env,
        text=True,
        capture_output=True,
    )


result_summary = {}
try:
    build = subprocess.run(
        ["bash", "make.sh"],
        cwd=ops_dir,
        text=True,
        capture_output=True,
    )
    build_text = (build.stdout or "") + "\n" + (build.stderr or "")
    highlights = re.findall(
        r"(?im)^(?:error:.*|.*undefined symbol.*|.*No module named.*|.*nvcc.*|.*failed.*|.*warning:.*)$",
        build_text,
    )
    probe = probe_model_import()
    result_summary = {
        "ops_build_returncode": build.returncode,
        "ops_build_highlights": highlights[:40],
        "ops_probe_returncode": probe.returncode,
        "ops_probe_stdout": probe.stdout.strip(),
        "ops_probe_stderr": probe.stderr.strip(),
        "ops_mode": "cuda_extension" if build.returncode == 0 and probe.returncode == 0 else "python_fallback",
    }
except Exception:
    result_summary = {
        "ops_mode": "python_fallback",
        "ops_error": traceback.format_exc(),
    }

state.update(result_summary)
STATE_PATH.write_text(json.dumps(state, indent=2), encoding="utf-8")
print(json.dumps(result_summary, indent=2))

In [ ]:
import json
import os
import shutil
import sys
import traceback
from pathlib import Path

STATE_PATH = Path("/kaggle/working/12a_wildtrack_mvdetr_state.json")
state = json.loads(STATE_PATH.read_text(encoding="utf-8"))
repo_dir = Path(state["repo_dir"])
target_root = Path.home() / "Data" / "Wildtrack"
required_dirs = ["Image_subsets", "annotations_positions", "calibrations"]


def is_wildtrack_root(path: Path) -> bool:
    return path.exists() and all((path / name).exists() for name in required_dirs)


def find_wildtrack_root() -> Path | None:
    for candidate in [Path(p) for p in state.get("wildtrack_candidates", [])]:
        if is_wildtrack_root(candidate):
            return candidate
    search_root = Path("/kaggle/input")
    if not search_root.exists():
        return None
    queue = [(search_root, 0)]
    while queue:
        current, depth = queue.pop(0)
        if is_wildtrack_root(current):
            return current
        if depth >= 4:
            continue
        try:
            children = [child for child in current.iterdir() if child.is_dir()]
        except Exception:
            continue
        for child in children:
            queue.append((child, depth + 1))
    return None


def replace_path(dst: Path, src: Path):
    if dst.exists() or dst.is_symlink():
        if dst.is_symlink() or dst.is_file():
            dst.unlink()
        else:
            shutil.rmtree(dst)
    try:
        dst.symlink_to(src, target_is_directory=src.is_dir())
    except OSError:
        if src.is_dir():
            shutil.copytree(src, dst)
        else:
            shutil.copy2(src, dst)


prep_summary = {}
try:
    source_root = find_wildtrack_root()
    if source_root is None:
        raise FileNotFoundError("Could not locate a WILDTRACK root containing Image_subsets, annotations_positions, and calibrations")

    target_root.parent.mkdir(parents=True, exist_ok=True)
    target_root.mkdir(parents=True, exist_ok=True)
    for name in required_dirs:
        replace_path(target_root / name, source_root / name)
    rectangles = source_root / "rectangles.pom"
    if rectangles.exists():
        replace_path(target_root / "rectangles.pom", rectangles)

    sys.path.insert(0, str(repo_dir))
    from multiview_detector.datasets import Wildtrack

    dataset = Wildtrack(str(target_root))
    prep_summary = {
        "data_ready": True,
        "wildtrack_source_root": str(source_root),
        "wildtrack_target_root": str(target_root),
        "target_entries": sorted(path.name for path in target_root.iterdir()),
        "dataset_probe": {
            "num_cam": int(dataset.num_cam),
            "num_frame": int(dataset.num_frame),
            "img_shape": list(dataset.img_shape),
            "worldgrid_shape": list(dataset.worldgrid_shape),
        },
    }
except Exception:
    prep_summary = {
        "data_ready": False,
        "data_prep_error": traceback.format_exc(),
    }

state.update(prep_summary)
STATE_PATH.write_text(json.dumps(state, indent=2), encoding="utf-8")
print(json.dumps(prep_summary, indent=2))

In [ ]:
import json
import os
import subprocess
import sys
import traceback
from pathlib import Path

STATE_PATH = Path("/kaggle/working/12a_wildtrack_mvdetr_state.json")
state = json.loads(STATE_PATH.read_text(encoding="utf-8"))
repo_dir = Path(state["repo_dir"])
artifact_root = Path(state["artifact_root"])

train_summary = {}
try:
    if not state.get("repo_ready"):
        raise RuntimeError("Repository setup did not complete successfully")
    if not state.get("data_ready"):
        raise RuntimeError("Dataset preparation did not complete successfully")

    epochs = int(os.environ.get("MVDETR_EPOCHS", "10"))
    num_workers = int(os.environ.get("MVDETR_NUM_WORKERS", "2"))
    batch_size = int(os.environ.get("MVDETR_BATCH_SIZE", "1"))
    cmd = [
        sys.executable,
        "main.py",
        "-d",
        "wildtrack",
        "--world_feat",
        "deform_trans",
        "--use_mse",
        "0",
        "--arch",
        "resnet18",
        "--epochs",
        str(epochs),
        "-j",
        str(num_workers),
        "-b",
        str(batch_size),
    ]

    artifact_root.mkdir(parents=True, exist_ok=True)
    stdout_path = artifact_root / "training_stdout.log"
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["OMP_NUM_THREADS"] = "1"
    env["PYTHONPATH"] = f"{repo_dir}:{env.get('PYTHONPATH', '')}" if env.get("PYTHONPATH") else str(repo_dir)

    print("+", " ".join(cmd))
    with stdout_path.open("w", encoding="utf-8") as log_file:
        proc = subprocess.Popen(
            cmd,
            cwd=repo_dir,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="")
            log_file.write(line)
        returncode = proc.wait()

    log_root = repo_dir / "logs" / "wildtrack"
    run_dirs = sorted([path for path in log_root.iterdir() if path.is_dir()], key=lambda path: path.stat().st_mtime) if log_root.exists() else []
    latest_run = run_dirs[-1] if run_dirs else None
    train_summary = {
        "train_command": cmd,
        "train_returncode": returncode,
        "training_stdout_log": str(stdout_path),
        "latest_run": str(latest_run) if latest_run else None,
        "latest_run_exists": bool(latest_run and latest_run.exists()),
    }
except Exception:
    train_summary = {
        "train_returncode": None,
        "training_error": traceback.format_exc(),
    }

state.update(train_summary)
STATE_PATH.write_text(json.dumps(state, indent=2), encoding="utf-8")
print(json.dumps(train_summary, indent=2))

In [ ]:
import json
import shutil
import traceback
import zipfile
from pathlib import Path

STATE_PATH = Path("/kaggle/working/12a_wildtrack_mvdetr_state.json")
state = json.loads(STATE_PATH.read_text(encoding="utf-8"))
artifact_root = Path(state["artifact_root"])
latest_run = Path(state["latest_run"]) if state.get("latest_run") else None

artifact_summary = {}
try:
    if latest_run is None or not latest_run.exists():
        raise FileNotFoundError("No MVDeTr run directory was found under logs/wildtrack")

    export_dir = artifact_root / latest_run.name
    export_dir.parent.mkdir(parents=True, exist_ok=True)
    if export_dir.exists():
        shutil.rmtree(export_dir)
    shutil.copytree(latest_run, export_dir)

    zip_path = artifact_root / f"{latest_run.name}.zip"
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for path in export_dir.rglob("*"):
            if path.is_file():
                zf.write(path, path.relative_to(export_dir))

    important = {
        "checkpoint": export_dir / "MultiviewDetector.pth",
        "detections": export_dir / "test.txt",
        "train_log": export_dir / "log.txt",
        "curve": export_dir / "learning_curve.jpg",
    }
    artifact_summary = {
        "artifact_dir": str(export_dir),
        "artifact_zip": str(zip_path),
        "important_files": {name: str(path) for name, path in important.items() if path.exists()},
        "artifact_entries": sorted(path.name for path in export_dir.iterdir()),
    }
except Exception:
    artifact_summary = {
        "artifact_error": traceback.format_exc(),
    }

state.update(artifact_summary)
STATE_PATH.write_text(json.dumps(state, indent=2), encoding="utf-8")
print(json.dumps(artifact_summary, indent=2))